# Graph-Enriched Search via MCP

This notebook extends vector search with **graph traversal** to pull in surrounding context for each match. Instead of returning isolated chunks, the agent follows graph relationships to include neighboring chunks, document metadata, and connected entities.

**Learning Objectives:**
- Combine vector similarity search with Cypher graph traversal through MCP
- Compare results between pure vector search and graph-enriched search
- Understand how graph context improves retrieval quality

**How it works:**
1. The notebook embeds the query using Bedrock Nova (same client-side embedding as [01_vector_search_mcp.ipynb](01_vector_search_mcp.ipynb))
2. The agent runs a vector search to find matching chunks via MCP
3. For each match, the agent traverses graph relationships to gather additional context:
   - Previous and next chunks (for a wider text window)
   - Source document metadata
   - Connected entities (companies, risk factors, etc.)
4. The enriched results give the LLM more context for generating answers

This mirrors the `VectorCypherRetriever` from the neo4j-graphrag library, but executed through MCP.

**Prerequisites:**
- `../CONFIG.txt` configured with `MODEL_ID`, `REGION`, `MCP_GATEWAY_URL`, and `MCP_ACCESS_TOKEN`
- Data loaded into Neo4j with vector embeddings on Chunk nodes

In [ ]:
%pip install strands-agents strands-agents-tools mcp httpx -q

## 1. Configuration and Imports

In [ ]:
import json
import os

import boto3
from dotenv import load_dotenv
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient

load_dotenv("../CONFIG.txt")

MODEL_ID = os.getenv("MODEL_ID")
REGION = os.getenv("REGION", "us-east-1")
MCP_GATEWAY_URL = os.getenv("MCP_GATEWAY_URL")
MCP_ACCESS_TOKEN = os.getenv("MCP_ACCESS_TOKEN")

# Embedding helper
NOVA_MODEL_ID = "amazon.nova-2-multimodal-embeddings-v1:0"
EMBEDDING_DIMENSIONS = 1024

bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)


def get_embedding(text: str) -> list[float]:
    """Generate an embedding vector for the given text using Bedrock Nova."""
    request_body = {
        "taskType": "SINGLE_EMBEDDING",
        "singleEmbeddingParams": {
            "embeddingPurpose": "GENERIC_INDEX",
            "embeddingDimension": EMBEDDING_DIMENSIONS,
            "text": {
                "truncationMode": "END",
                "value": text,
            },
        },
    }
    response = bedrock_runtime.invoke_model(
        modelId=NOVA_MODEL_ID,
        body=json.dumps(request_body),
    )
    result = json.loads(response["body"].read())
    return result["embeddings"][0]["embedding"]


print(f"Model:     {MODEL_ID}")
print(f"Region:    {REGION}")
print("Configuration loaded!")

## 2. Connect to MCP and Create Agents

We create **three agents** with different system prompts to compare retrieval approaches:

1. **Vector-only agent** — returns just the matched chunk text and score (same as the previous notebook)
2. **Graph-enriched agent** — follows graph relationships to include document metadata and neighboring chunks
3. **Entity-enriched agent** — traverses to the company that filed the document, plus products and risk factors

### Entity Relationships in the Graph

The knowledge graph connects chunks to companies through document ownership:

```
(Chunk)-[:FROM_DOCUMENT]->(Document)<-[:FILED]-(Company)  the company that filed the document
(Product)-[:FROM_CHUNK]->(Chunk)     products mentioned in a chunk
(Company)-[:FACES_RISK]->(RiskFactor)  risks associated with companies
```

By traversing these relationships, the agent can augment each vector match with structured entity data.

In [ ]:
VECTOR_ONLY_PROMPT = """You are a retrieval assistant that performs vector search against a Neo4j database.

When given a query embedding, use this Cypher to find similar chunks:

CALL db.index.vector.queryNodes('chunkEmbeddings', $top_k, $embedding)
YIELD node, score
RETURN node.text AS text, score
ORDER BY score DESC

Return the chunk text and similarity score. Use the exact embedding provided."""

GRAPH_ENRICHED_PROMPT = """You are a retrieval assistant that performs graph-enriched vector search against a Neo4j database containing SEC 10-K filing data.

When given a query embedding, use this Cypher to find similar chunks WITH graph context:

CALL db.index.vector.queryNodes('chunkEmbeddings', $top_k, $embedding)
YIELD node, score
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
OPTIONAL MATCH (node)-[:NEXT_CHUNK]->(next:Chunk)
OPTIONAL MATCH (prev:Chunk)-[:NEXT_CHUNK]->(node)
WITH node, doc, score,
     CASE WHEN prev IS NOT NULL THEN prev.text ELSE '' END AS prev_text,
     CASE WHEN next IS NOT NULL THEN next.text ELSE '' END AS next_text
RETURN node.text AS text,
       score,
       doc.name AS document,
       prev_text AS previous_chunk,
       next_text AS next_chunk
ORDER BY score DESC

This query:
1. Finds the most similar chunks via vector search
2. Traverses FROM_DOCUMENT to get the source document name
3. Follows NEXT_CHUNK relationships to get neighboring chunk text
4. Returns the enriched context alongside each match

Use the exact embedding provided. For each result, show:
- Similarity score
- Source document name
- The matched chunk text
- Context from neighboring chunks (if available)"""

ENTITY_ENRICHED_PROMPT = """You are a retrieval assistant that performs entity-enriched vector search against a Neo4j database containing SEC 10-K filing data.

When given a query embedding, use this Cypher to find similar chunks WITH entity context:

CALL db.index.vector.queryNodes('chunkEmbeddings', $top_k, $embedding)
YIELD node, score
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
OPTIONAL MATCH (doc)<-[:FILED]-(company:Company)
OPTIONAL MATCH (company)-[:FACES_RISK]->(risk:RiskFactor)
WITH node, doc, score,
     collect(DISTINCT company.name) AS companies,
     collect(DISTINCT risk.name)[0..5] AS risks
OPTIONAL MATCH (product:Product)-[:FROM_CHUNK]->(node)
WITH node, doc, score, companies, risks,
     collect(DISTINCT product.name)[0..5] AS products
RETURN node.text AS text,
       score,
       doc.name AS document,
       companies,
       risks,
       products
ORDER BY score DESC

This query:
1. Finds the most similar chunks via vector search
2. Traverses FROM_DOCUMENT to get the source document
3. Follows FILED to find the company that filed the document
4. Follows FACES_RISK from companies to find their risk factors

Use the exact embedding provided. For each result, show:
- Similarity score
- Source document name
- The matched chunk text
- Companies, products, and risk factors connected to the chunk"""

bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    temperature=0,
)

mcp_client = MCPClient(
    lambda: streamablehttp_client(
        url=MCP_GATEWAY_URL,
        headers={"Authorization": f"Bearer {MCP_ACCESS_TOKEN}"},
    )
)

print(f"Model: {MODEL_ID}")
print("MCP client and prompts configured.")

## 3. Compare: Vector-Only vs Graph-Enriched vs Entity-Enriched

Run the same query through all three agents to see how each level of graph traversal enriches the results.

In [ ]:
def compare_search(query: str, top_k: int = 3):
    """Run the same query through all three agents and display results."""
    embedding = get_embedding(query)

    message = (
        f"Run a vector search for the following query. Use top_k={top_k}.\n\n"
        f"Query: {query}\n\n"
        f"Embedding (use this exact array in the Cypher query):\n{json.dumps(embedding)}"
    )

    print(f'Query: "{query}"')
    print("=" * 60)

    with mcp_client:
        tools = mcp_client.list_tools_sync()

        # Vector-only search
        print("\n--- VECTOR-ONLY RESULTS ---\n")
        vector_agent = Agent(
            model=bedrock_model,
            system_prompt=VECTOR_ONLY_PROMPT,
            tools=tools,
        )
        print(vector_agent(message))

        # Graph-enriched search
        print("\n\n--- GRAPH-ENRICHED RESULTS ---\n")
        graph_agent = Agent(
            model=bedrock_model,
            system_prompt=GRAPH_ENRICHED_PROMPT,
            tools=tools,
        )
        print(graph_agent(message))

        # Entity-enriched search
        print("\n\n--- ENTITY-ENRICHED RESULTS ---\n")
        entity_agent = Agent(
            model=bedrock_model,
            system_prompt=ENTITY_ENRICHED_PROMPT,
            tools=tools,
        )
        print(entity_agent(message))

In [ ]:
compare_search("What are the key risk factors mentioned in Apple's 10-K filing?")

In [ ]:
compare_search("What financial metrics indicate company performance?")

## 4. Graph-Enriched Q&A

Now use the entity-enriched agent as a full question-answering pipeline. The agent finds relevant chunks via vector search, gathers graph context, and uses all of that to generate a comprehensive answer.

In [ ]:
QA_PROMPT = """You are a financial analysis assistant with access to a Neo4j knowledge graph containing SEC 10-K filing data.

You have MCP tools to execute Cypher queries. Use entity-enriched vector search to answer questions:

CALL db.index.vector.queryNodes('chunkEmbeddings', $top_k, $embedding)
YIELD node, score
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
OPTIONAL MATCH (doc)<-[:FILED]-(company:Company)
OPTIONAL MATCH (company)-[:FACES_RISK]->(risk:RiskFactor)
WITH node, doc, score,
     collect(DISTINCT company.name) AS companies,
     collect(DISTINCT risk.name)[0..5] AS risks
OPTIONAL MATCH (product:Product)-[:FROM_CHUNK]->(node)
WITH node, doc, score, companies, risks,
     collect(DISTINCT product.name)[0..5] AS products
RETURN node.text AS text, score, doc.name AS document,
       companies, risks, products
ORDER BY score DESC

After retrieving results, synthesize a clear answer based on the retrieved context.
Include the companies, products, and risk factors found. Cite the source documents.
Use the exact embedding provided."""


def ask(query: str, top_k: int = 5):
    """Ask a question using graph-enriched vector search for context."""
    embedding = get_embedding(query)

    message = (
        f"Answer this question using graph-enriched vector search with top_k={top_k}.\n\n"
        f"Question: {query}\n\n"
        f"Embedding:\n{json.dumps(embedding)}"
    )

    print(f'Question: "{query}"')
    print("-" * 60)

    with mcp_client:
        tools = mcp_client.list_tools_sync()
        qa_agent = Agent(
            model=bedrock_model,
            system_prompt=QA_PROMPT,
            tools=tools,
        )
        response = qa_agent(message)
        print(f"\n{response}")
        return response

In [ ]:
result = ask("What are the key risk factors mentioned in Apple's 10-K filing?")

In [ ]:
result = ask("Which companies face cybersecurity-related risks?")

## Summary

You compared three retrieval approaches, all executed through MCP:

| Approach | What the LLM Receives |
|----------|----------------------|
| **Vector-only** | The matched chunk text and similarity score |
| **Graph-enriched** | The matched chunk + document name + neighboring chunk text |
| **Entity-enriched** | The matched chunk + document + companies, products, risk factors |

Each level of graph traversal adds more context. The entity-enriched approach is the most powerful because it connects unstructured text (chunks) to structured data (entity nodes and their relationships), giving the LLM both the raw filing text and the knowledge graph's understanding of what's in it.

### A Note on Embedding Handling

The current approach serializes the full 1024-dimensional embedding (~8-10KB of JSON) into the agent's message. This works but is inefficient — it consumes context window tokens with numeric data the LLM cannot meaningfully interpret. In the next notebook, you will see a cleaner approach: wrapping embedding generation and MCP query execution inside a custom `@tool` function, so the agent calls `vector_search("Apple products")` instead of receiving a giant float array.

---

**Next:** [Fulltext and Hybrid Search via MCP](03_fulltext_hybrid_search_mcp.ipynb)